In [ ]:
import kagglehub


path = kagglehub.dataset_download("vipoooool/new-plant-diseases-dataset")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/vipoooool/new-plant-diseases-dataset


In [ ]:

import os

base_dir = os.path.join(path, 'New Plant Diseases Dataset(Augmented)', 'New Plant Diseases Dataset(Augmented)')
train_dir = os.path.join(base_dir, 'train')
valid_dir = os.path.join(base_dir, 'valid')

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

batch_size = 512  
target_size = (224, 224)

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    train_dir,
    labels='inferred',
    label_mode='categorical',
    image_size=target_size,
    batch_size=batch_size,
    shuffle=True,
    seed=42
)

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomTranslation(0.1, 0.1)
])

def preprocess_train(image, label):
    image = data_augmentation(image, training=True)
    image = preprocess_input(image)
    return image, label

train_ds = train_ds.map(preprocess_train, num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.prefetch(buffer_size=tf.data.AUTOTUNE)

valid_ds = tf.keras.preprocessing.image_dataset_from_directory(
    valid_dir,
    labels='inferred',
    label_mode='categorical',
    image_size=target_size,
    batch_size=batch_size,
    shuffle=False
)

def preprocess_valid(image, label):
    image = preprocess_input(image)
    return image, label

valid_ds = valid_ds.map(preprocess_valid, num_parallel_calls=tf.data.AUTOTUNE)
valid_ds = valid_ds.prefetch(buffer_size=tf.data.AUTOTUNE)

Found 70295 files belonging to 38 classes.
Found 17572 files belonging to 38 classes.


In [16]:
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
import os


num_classes = len(os.listdir(train_dir))

with strategy.scope():
    base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224,224,3))
    base_model.trainable = False

    model = Sequential([
        base_model,
        GlobalAveragePooling2D(),
        Dense(256, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='softmax', dtype='float32')  
    ])

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint

checkpoint = ModelCheckpoint(
    'best_plant_model.keras',  
    monitor='val_accuracy',
    mode='max',
    save_best_only=True,        
    verbose=1
)

In [ ]:
history = model.fit(
    train_ds,                             
    epochs=8,
    validation_data=valid_ds,             
    callbacks=[checkpoint]
)

INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
Epoch 1/8
INFO:tensorflow:Collective all_reduce tensors: 4 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1
138/138 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.4980 - loss: 1.8605
Epoch 1: val_accuracy improved from -inf to 0.88982, saving model to best_plant_model.keras
138/138 ━━━━━━━━━━━━━━━━━━━━ 619s 4s/step - accuracy: 0.4993 - loss: 1.8552 - val_accuracy: 0.8898 - val_loss: 0.3836
Epoch 2/8
138/138 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.8463 - loss: 0.4941
Epoch 2: val_accuracy improved from 0.88982 to 0.91862, saving model to best_plant_model.keras
138/138 ━━━━━━━━━━━━━━━━━━━━ 590s 4s/step - accuracy: 0.8464 - loss: 0.4938 - val_accur

In [ ]:

base_model.trainable = True
for layer in base_model.layers[:100]:
    layer.trainable = False


with strategy.scope():
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )


history_fine = model.fit(
    train_ds,
    epochs=5,
    validation_data=valid_ds,
    callbacks=[checkpoint]
)

model.save('plant_disease_mobilenet.keras')




Epoch 1/5
INFO:tensorflow:Collective all_reduce tensors: 58 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1
138/138 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.6727 - loss: 1.2275
Epoch 1: val_accuracy did not improve from 0.94645
138/138 ━━━━━━━━━━━━━━━━━━━━ 611s 4s/step - accuracy: 0.6733 - loss: 1.2247 - val_accuracy: 0.9258 - val_loss: 0.2279
Epoch 2/5
138/138 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.8707 - loss: 0.4022
Epoch 2: val_accuracy did not improve from 0.94645
138/138 ━━━━━━━━━━━━━━━━━━━━ 575s 4s/step - accuracy: 0.8708 - loss: 0.4020 - val_accuracy: 0.9147 - val_loss: 0.2626
Epoch 3/5
138/138 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.9022 - loss: 0.3006
Epoch 3: val_accuracy did not improve from 0.94645
138/138 ━━━━━━━━━━━━━━━━━━━━ 574s 4s/step - accuracy: 0.9022 - loss: 0.3005 - val_accuracy: 0.9144 - val_loss: 0.2676
Epoch 4/5
138/138 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.9158 - loss: 0.2

In [27]:
import json
import os

class_names_list = sorted(os.listdir(train_dir))
class_names = {str(i): name for i, name in enumerate(class_names_list)}

with open('class_names.json', 'w') as f:
    json.dump(class_names, f)

print("Class indices saved successfully.")
print(f"Number of classes: {len(class_names)}")

Class indices saved successfully.
Number of classes: 38
